In [1]:
%pwd

'c:\\Users\\mnsnn\\Documents\\Learn Advanced\\ML - MLOps\\datascienceproject\\research'

In [2]:
import os
os.chdir('../')

%pwd

'c:\\Users\\mnsnn\\Documents\\Learn Advanced\\ML - MLOps\\datascienceproject'

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI')
MLFLOW_TRACKING_USERNAME = os.getenv('MLFLOW_TRACKING_USERNAME')
MLFLOW_TRACKING_PASSWORD = os.getenv('MLFLOW_TRACKING_PASSWORD')

In [12]:
# entity/config_entity.py

from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelEvaluationConfig:
	root_dir : Path
	test_data_path : Path
	model_path : Path
	metric_file_name : Path
	target : str
	all_params : dict

In [14]:
# config/configutaion.py

from src.datascienceproject.constants import PARAMS_FILE_PATH
from src.datascienceproject.constants import SCHEMA_FILE_PATH
from src.datascienceproject.constants import CONFIG_FILE_PATH

from src.datascienceproject.utils.common import read_yaml, save_json, create_directories

class ConfigurationManager:
    
	def __init__(self,
		params_file_path = PARAMS_FILE_PATH,
		schema_file_path = SCHEMA_FILE_PATH,
		config_file_path = CONFIG_FILE_PATH):

		self.config = read_yaml(config_file_path)
		self.params = read_yaml(params_file_path)
		self.schema = read_yaml(schema_file_path)

		# Creating Artifacts root dir
		create_directories([self.config.artifacts_root])

	def get_model_evaluation_configs(self):

		config = self.config.model_evaluation
		schema = self.schema
		params = self.params
		create_directories([config.root_dir])

		model_evaluation_config = ModelEvaluationConfig(
			root_dir = config.root_dir,
			test_data_path = config.test_data_path,
			model_path = config.model_path,
			metric_file_name = config.metric_file_name,
			target = schema.TARGET_COLUMN.name,
			all_params = params.ElasticNet
		)

		return model_evaluation_config
		

In [ ]:
# components/model_evaluation.py
import os
from dotenv import load_dotenv
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

from src.datascienceproject import logger

load_dotenv()

mlflow_tracking_uri = os.getenv('MLFLOW_TRACKING_URI')
mlflow_tracking_username = os.getenv('MLFLOW_TRACKING_USERNAME')
mlflow_tracking_password = os.getenv('MLFLOW_TRACKING_PASSWORD')

class ModelEvaluation:
    
	def __init__(self, config:ModelEvaluationConfig) -> ModelEvaluationConfig:
		self.config = config

	def eval_metrics(self, actual, pred):

		rmse = np.sqrt(mean_squared_error(actual, pred))
		mae = mean_absolute_error(actual, pred)
		r2 = r2_score(actual, pred)

		return rmse, mae, r2
	
	def evaluate(self):
		
		test_data = pd.read_csv(self.config.test_data_path)
		model = joblib.load(self.config.model_path)

		X_test = test_data.drop([self.config.target], axis=1)
		y_test = test_data[[self.config.target]]

		mlflow.set_tracking_uri(mlflow_tracking_uri)
		tracking_uri_type_store = urlparse(mlflow.get_tracking_uri()).scheme

		with mlflow.start_run():

			y_pred = model.predict(X_test)
			rmse, mae, r2 = self.eval_metrics(y_test, y_pred)

			scores = {'rmse': rmse, 'mae': mae, 'r2': r2}
			save_json(save_path=Path(self.config.metric_file_name), data=scores)

			mlflow.log_params(self.config.all_params)
			mlflow.log_metric('rmse', rmse)
			mlflow.log_metric('mae', mae)
			mlflow.log_metric('r2', r2)

			if tracking_uri_type_store != 'file':
				mlflow.sklearn.log_model(model, 'model', registered_model_name='ElasticNet - Production')
			else:
				mlflow.sklearn.log_model(model, 'model')


In [28]:
config = ConfigurationManager()
model_evaluation_config = config.get_model_evaluation_configs()
model_evaluation = ModelEvaluation(config=model_evaluation_config)
model_evaluation.evaluate()

2026-05-27 08:19:01,390 : INFO : YAML file config\config.yaml loaded successfully
2026-05-27 08:19:01,394 : INFO : YAML file params.yaml loaded successfully
2026-05-27 08:19:01,401 : INFO : YAML file schema.yaml loaded successfully
2026-05-27 08:19:01,405 : INFO : Created directory at artifacts
2026-05-27 08:19:01,408 : INFO : Created directory at artifacts/model_evaluation
2026-05-27 08:19:02,097 : INFO : JSON file saved at: artifacts\model_evaluation\metrics.json


2026/05/27 08:19:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/27 08:19:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'ElasticNet - Production'.
2026/05/27 08:19:30 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticNet - Production, version 1
Created version '1' of model 'ElasticNet - Production'.


🏃 View run resilient-dolphin-61 at: https://dagshub.com/nishankode/MLPipeline.mlflow/#/experiments/0/runs/87c8a69000024d75858a70766fb56b4f
🧪 View experiment at: https://dagshub.com/nishankode/MLPipeline.mlflow/#/experiments/0
